# Mission Analysis
### Roadrunner propulsion reverse-engineering study · notebook 12 of 15

**Purpose.** Fly the sample mission of notebook 8 with the engine's own thrust and part-throttle
fuel consumption from the off-design cycle analysis (notebook 10). It answers:

> *How much fuel does the mission burn once fuel consumption depends on flight condition and
> throttle, and what does that leave for the empty vehicle?*

The constant TSFC of the preliminary analysis is replaced by the cycle's SFC at maximum power
for climb and acceleration, and at the throttle each segment needs (drag over available thrust;
hover at $T=W$) for level flight, loiter, launch and landing. The thrust loading is the one the
constraint analysis demands, so notebooks 11 and 12 describe one solution. Everything else is
as in notebook 8 (ADR-0018, ADR-0029, ADR-0033).

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from roadrunner_re import analysis_outputs as ao
from roadrunner_re import reports
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.engine_model import CycleEngine
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots import analysis as plots

paths = nb_setup()
preliminary_inputs = load_analysis_inputs()                       # algebraic lapse, constant TSFC
off_design = yaml.safe_load((paths.out / "off_design.yaml").read_text(encoding="utf-8"))
inputs = preliminary_inputs.with_engine(CycleEngine(off_design))   # pyCycle lapse and part-throttle SFC
preliminary = solve_design_point(preliminary_inputs)               # the single call site (ADR-0029)
design = solve_design_point(inputs)
sweep = ao.takeoff_mass_sweep(inputs, ao.sweep_masses(inputs))

---
## 1 · Weight fractions and fuel

In [ ]:
print(reports.weight_fraction_table(design))
print(f"\nconverged in {design.iterations} iterations")
fig = plots.mission_weights_figure(design, paths.figures / "mission_weights_cycle.png")

---
## 2 · Against the preliminary analysis

In [ ]:
comparison = ao.mission_comparison(preliminary, design)
print(reports.fuel_comparison_table(comparison))
fig = plots.fuel_comparison_figure(preliminary, design, paths.figures / "fuel_comparison.png")

---
## 3 · What the take-off mass leaves for the vehicle

As in notebook 8 the take-off mass is swept, not solved: there is no structural build-up
(ADR-0008), so the implied empty mass is reported next to the engine mass from the catalogue
power law.

In [ ]:
m = design.mission
print(f"take-off mass          {m.takeoff_mass_kg:7.2f} kg")
print(f"payload                {m.payload_mass_kg:7.2f} kg")
print(f"fuel incl. reserve     {m.total_fuel_kg:7.2f} kg")
print(f"implied empty mass     {m.implied_empty_mass_kg:7.2f} kg")
print(f"thrust per engine      {design.thrust_per_engine_N:7.0f} N (installed = uninstalled, no installation loss yet)")
print(f"propulsion (2 engines) {design.propulsion_mass_kg:7.2f} kg = {design.propulsion_fraction_of_empty:.0%} of implied empty mass")

In [ ]:
fig = plots.mass_sweep_figure(sweep, paths.figures / "mass_sweep_cycle.png")

---
## 4 · Handoff

In [ ]:
content = ao.mission_out(inputs, design, sweep, notebook="mission_analysis", comparison=comparison)
handoff = ao.write_handoff(content, paths.out / "mission.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **Mission fuel is 19.9 kg, 55 % more than the preliminary 12.8 kg** (21.9 kg with the 10 % reserve
  against 14.1 kg). The loiter dominates: 11.9 kg against 7.0 kg (ratio 1.71), because the engine runs at
  about 28 % of maximum thrust, where SFC is poor, and the preliminary constant TSFC ignored that. The
  dash and return, which fly at higher throttle, are 1.4 to 1.6 times the preliminary fuel.
- **Launch and landing hardly change** (0.32 and 0.78 kg), and the descent, a fixed weight fraction, is
  unchanged.
- **What it leaves for the vehicle.** At 85 kg take-off mass with a 10 kg payload the implied empty mass
  falls from 60.9 kg to 53.1 kg; the two engines are 14.5 kg of it. This is a residual, not a build-up
  (ADR-0008), so it says only that the fuel the mission needs takes a larger share of the 85 kg.
- **Standing caveats.** Combustion efficiency is constant with throttle, so part-throttle fuel flow is
  probably understated and the true fuel is higher still; the hover SFC uses the sea-level-static
  penalty curve and the flight SFC the mean of the three flight curves; thrust is uninstalled.
- **Next.** Engine selection (notebook 13) tests catalogue engines against this thrust and fuel
  requirement.